In [ ]:
import os; os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == "notebooks" else None

# 02 - Baselines: PI controller and stored RL runs

Reproduces the PI-controller benchmark number, loads the results/learning-curve files
written by `scripts/evaluate.py` and `scripts/train.py`, and re-evaluates the best
stored RL checkpoint to check it is reproducible. Every cell that reads a file under
`data/` prints a clear message and continues (does not raise) if that file is missing -
in that case, run `bash scripts/reproduce.sh` first and re-run this notebook.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from rl_tokamak.env import EnvConfig, RampupEnv
from rl_tokamak.controllers import PIController
from rl_tokamak.evaluate import run_controller, run_policy
from rl_tokamak.policies import load_policy
from rl_tokamak.plotting import plot_episodes, plot_learning_curves

REPRODUCE_MSG = "run bash scripts/reproduce.sh first"

## (a) Reproduce the PI-controller number

`PIController` re-implements the paper's PI policy (Section "Policies", arXiv 2510.11283)
on a fresh environment. The ITER-hybrid TORAX dynamics are deterministic, so this run
should land within 0.01 of the published 3.79.

In [ ]:
env = RampupEnv(EnvConfig())
pi_result = run_controller(env, PIController())
print("PI benchmark_return:", pi_result["benchmark_return"], " failed:", pi_result["failed"])

assert abs(pi_result["benchmark_return"] - 3.79) < 0.01, pi_result["benchmark_return"]
print("matches paper PI return (3.79) within 0.01")

In [ ]:
classical_path = Path("data/results/classical.json")
if classical_path.exists():
    classical = json.loads(classical_path.read_text())
    stored = classical["pi"]["benchmark_return"]
    print("stored data/results/classical.json PI return:", stored, " (paper", classical["pi"]["paper"], ")")
    print("this run vs stored: abs diff =", abs(pi_result["benchmark_return"] - stored))
else:
    print(f"{classical_path} not found - {REPRODUCE_MSG}")

## (b) Results summary table

`data/results/summary.json` (written by `scripts/evaluate.py --summary`) collects every
classical and RL result into one table: `policy`, `group`, `return`, `paper`, `env_steps`,
`minutes`, and a few end-of-episode physics numbers.

In [ ]:
summary_path = Path("data/results/summary.json")
summary_df = None
if summary_path.exists():
    summary_df = pd.DataFrame(json.loads(summary_path.read_text())).sort_values("return", ascending=False)
else:
    print(f"{summary_path} not found - {REPRODUCE_MSG}")
summary_df

## (c) Re-evaluate the best stored RL checkpoint

Picks the online run (`algo` in `ppo`/`sac`/`mbpo`) with the highest stored
`result.json["benchmark_return"]`, reloads its policy with `load_policy`, and re-runs
one deterministic episode. The ITER-hybrid environment and every stored policy here are
deterministic, so the fresh return should match the stored one to numerical precision.

In [ ]:
ONLINE_ALGOS = {"ppo", "sac", "mbpo"}

best_run, best_return = None, -float("inf")
for result_path in sorted(Path("data/runs").glob("*/result.json")):
    cfg = json.loads((result_path.parent / "config.json").read_text())
    if cfg["algo"] not in ONLINE_ALGOS:
        continue
    res = json.loads(result_path.read_text())
    if res["benchmark_return"] > best_return:
        best_run, best_return = result_path.parent, res["benchmark_return"]

if best_run is None:
    print(f"no online run result.json found under data/runs - {REPRODUCE_MSG}")
else:
    print("best online run:", best_run.name, " stored benchmark_return =", best_return)

In [ ]:
eval_env = None
if best_run is not None:
    env_cfg, policy_fn = load_policy(best_run)
    eval_env = RampupEnv(env_cfg)
    ep = run_policy(eval_env, policy_fn)
    print("re-evaluated benchmark_return:", ep["benchmark_return"])
    diff = abs(ep["benchmark_return"] - best_return)
    print("abs diff vs stored result.json:", diff)
    assert diff < 1e-6, diff
    print("matches stored result.json within 1e-6")
else:
    print(f"skipped - {REPRODUCE_MSG}")

## (d) Learning curves

`curve.json` (written during training) holds the deterministic-policy evaluation curve
for every online run. Plots all `ppo`/`sac`/`mbpo` runs found under `data/runs/` against
simulator steps used, with the PI (3.79) and open-loop (3.40) benchmark returns as
reference lines.

In [ ]:
online_runs = {}
for cfg_path in sorted(Path("data/runs").glob("*/config.json")):
    cfg = json.loads(cfg_path.read_text())
    if cfg["algo"] in ONLINE_ALGOS and (cfg_path.parent / "curve.json").exists():
        online_runs[cfg_path.parent.name] = cfg_path.parent

if online_runs:
    print("online runs with a curve:", sorted(online_runs))
    fig = plot_learning_curves(online_runs, x="env_steps", references={"PI": 3.79, "open-loop": 3.40})
else:
    print(f"no online runs with curve.json found under data/runs - {REPRODUCE_MSG}")
    fig = None
fig

## (e) Best run vs PI trajectory

Compares the final deterministic episode of the best online run (`final_episode.csv`,
written by `evaluate_run`) against the PI-controller reference trajectory
(`data/trajectories/pi.csv`, written by `scripts/evaluate.py --classical`).

In [ ]:
pi_csv = Path("data/trajectories/pi.csv")
best_episode_csv = best_run / "final_episode.csv" if best_run is not None else None

if best_episode_csv is not None and best_episode_csv.exists() and pi_csv.exists():
    fig2 = plot_episodes({best_run.name: best_episode_csv, "PI": pi_csv},
                         title=f"{best_run.name} vs PI controller")
else:
    missing = [p for p in (best_episode_csv, pi_csv) if p is None or not p.exists()]
    print(f"missing {missing} - {REPRODUCE_MSG}")
    fig2 = None
fig2

In [ ]:
env.close()
if eval_env is not None:
    eval_env.close()